In [ ]:
import os
import shutil
import tempfile
from datetime import datetime
from pathlib import Path

# === 🧠 SET YOUR FOLDER PATH HERE ===
SOURCE_FOLDER = Path("Pranjal/Prokka_Outputs").resolve()

# === MAIN FUNCTIONS ===

def make_copy(src: Path) -> Path:
    """Make a timestamped copy of the main folder."""
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    dst = src.with_name(f"{src.name}_copy_{ts}")
    if dst.exists():
        raise FileExistsError(f"Destination already exists: {dst}")
    print(f"📁 Copying folder:\n  from: {src}\n  to:   {dst}")
    shutil.copytree(src, dst)
    print("✅ Copy complete.\n")
    return dst

def is_ffn(filename: str) -> bool:
    return filename.lower().endswith('.ffn')

def sanitize_header(header: str, top_level_folder: str) -> str:
    """Replace spaces with underscores and prepend subfolder name."""
    h = header.replace(' ', '_')
    prefix = f"{top_level_folder}_"
    if not h.startswith(prefix):
        h = prefix + h
    return '>' + h

def top_level_subfolder_for_dir(root_copied: Path, dirpath: Path) -> str:
    """Return the first subfolder name under root_copied."""
    rel = os.path.relpath(dirpath, root_copied)
    if rel in ('.', ''):
        return root_copied.name
    parts = Path(rel).parts
    return parts[0]

def process_ffn_file(ffn_path: Path, top_level_folder: str):
    """Modify FASTA headers inside .ffn file."""
    print(f"  🧬 Processing .ffn file: {ffn_path}")
    dir_for_temp = ffn_path.parent
    with tempfile.NamedTemporaryFile('w', delete=False, dir=str(dir_for_temp), prefix=".tmp_ffn_", suffix=".ffn") as tmpf:
        tmp_path = Path(tmpf.name)
        with ffn_path.open('r') as rf:
            for line in rf:
                if line.startswith('>'):
                    header_body = line[1:].rstrip('\n\r')
                    new_header = sanitize_header(header_body, top_level_folder)
                    tmpf.write(new_header + '\n')
                else:
                    tmpf.write(line)
    # Replace file safely
    backup = ffn_path.with_suffix(ffn_path.suffix + '.bak')
    try:
        if backup.exists():
            backup.unlink()
        ffn_path.rename(backup)
        tmp_path.rename(ffn_path)
        backup.unlink()
    except Exception as e:
        print(f"❌ Error replacing file {ffn_path}: {e}")
        if backup.exists() and not ffn_path.exists():
            backup.rename(ffn_path)
        if tmp_path.exists():
            tmp_path.unlink()
        raise

def cleanup_non_ffn_files(root_copied: Path):
    """Delete all non-.ffn files in the copied directory."""
    print("🧹 Removing non-.ffn files in the copied tree...")
    for dirpath, dirnames, filenames in os.walk(root_copied):
        dirpath_p = Path(dirpath)
        for fname in filenames:
            if not is_ffn(fname):
                target = dirpath_p / fname
                try:
                    target.unlink()
                    print(f"  Deleted: {target}")
                except Exception as e:
                    print(f"  Failed to delete {target}: {e}")

def process_copied_tree(root_copied: Path):
    """Iterate through copied folder and process each .ffn file."""
    print("🧬 Modifying .ffn files...\n")
    for dirpath, dirnames, filenames in os.walk(root_copied):
        dirpath_p = Path(dirpath)
        top_level = top_level_subfolder_for_dir(root_copied, dirpath_p)
        for fname in filenames:
            if is_ffn(fname):
                ffn_path = dirpath_p / fname
                process_ffn_file(ffn_path, top_level)

# === 🚀 MAIN EXECUTION ===

if not SOURCE_FOLDER.exists() or not SOURCE_FOLDER.is_dir():
    raise FileNotFoundError(f"Source folder not found: {SOURCE_FOLDER}")

copied_root = make_copy(SOURCE_FOLDER)
cleanup_non_ffn_files(copied_root)
process_copied_tree(copied_root)

print("\n✅ All done!")
print(f"Processed copied folder is here:\n{copied_root}")

In [ ]:
from pathlib import Path

# === Set your final folder path here ===
ROOT = Path("Pranjal/Prokka_Outputs_copy").resolve()

# === Stop codons ===
STOP_CODONS = {"TAA", "TAG", "TGA"}

def fasta_reader(ffn_path):
    """Simple FASTA parser: yields (header, sequence) tuples."""
    header = None
    seq_lines = []
    with open(ffn_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header:
                    yield header, "".join(seq_lines)
                header = line
                seq_lines = []
            else:
                seq_lines.append(line.upper())  # store in uppercase
        if header:
            yield header, "".join(seq_lines)

def has_internal_stop(seq):
    """Return True if CDS has a stop codon before the last codon."""
    for i in range(0, len(seq) - 3, 3):  # exclude last codon
        codon = seq[i:i+3]
        if codon in STOP_CODONS:
            return True
    return False

def is_valid_cds(seq):
    """Return True if seq length divisible by 3 and no internal stop codons."""
    if len(seq) % 3 != 0:
        return False
    if has_internal_stop(seq):
        return False
    return True

def process_ffn_file(ffn_path):
    """Filter and overwrite .ffn file based on CDS validity."""
    valid_records = []
    total = 0
    removed = 0
    
    for header, seq in fasta_reader(ffn_path):
        total += 1
        if is_valid_cds(seq):
            valid_records.append((header, seq))
        else:
            removed += 1

    # Overwrite the same file with valid records
    with open(ffn_path, "w") as f:
        for header, seq in valid_records:
            f.write(f"{header}\n")
            # Write sequence in 70-char lines (FASTA style)
            for i in range(0, len(seq), 70):
                f.write(seq[i:i+70] + "\n")
    print(f"Processed {ffn_path} — kept {len(valid_records)}/{total}, removed {removed}")

def process_all_ffn(root_folder):
    """Walk through all subfolders and process each .ffn file."""
    ffn_files = list(root_folder.rglob("*.ffn"))
    if not ffn_files:
        print("⚠️ No .ffn files found under:", root_folder)
        return
    print(f"Found {len(ffn_files)} .ffn files. Starting processing...\n")
    for ffn in ffn_files:
        process_ffn_file(ffn)
    print("\n✅ All .ffn files processed successfully!")

# === Run the filtering ===
process_all_ffn(ROOT)

In [ ]:
import shutil
from pathlib import Path

# === 🧠 Set your folder paths ===
SOURCE = Path("/home/bioinfo/Desktop/Pratanu/Metabolic_Rate/Pranjal/Prokka_Outputs_copy").resolve()
DEST = SOURCE.with_name("Prokka_Outputs_copy_Concatenate")

STOP_CODONS = {"TAA", "TAG", "TGA"}

# === Utility functions ===

def make_copy(src: Path, dst: Path):
    """Create a full copy of the source directory."""
    if dst.exists():
        print(f"⚠️ Destination already exists, removing old copy: {dst}")
        shutil.rmtree(dst)
    print(f"📁 Copying folder...\n  from: {src}\n  to:   {dst}")
    shutil.copytree(src, dst)
    print("✅ Copy complete.\n")

def fasta_reader(ffn_path):
    """Simple FASTA parser that yields (header, sequence)."""
    header, seq_lines = None, []
    with open(ffn_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header:
                    yield header, "".join(seq_lines)
                header = line
                seq_lines = []
            else:
                seq_lines.append(line.upper())
        if header:
            yield header, "".join(seq_lines)

def remove_terminal_stop(seq):
    """Remove terminal stop codon if present."""
    if len(seq) < 3:
        return seq
    last_codon = seq[-3:]
    if last_codon in STOP_CODONS:
        return seq[:-3]
    return seq

def concatenate_cds_in_file(ffn_path):
    """Concatenate all CDS in order, removing terminal stops except the last."""
    seqs = [seq for _, seq in fasta_reader(ffn_path)]
    if not seqs:
        return ""
    concatenated = ""
    for i, seq in enumerate(seqs):
        if i < len(seqs) - 1:
            seq = remove_terminal_stop(seq)
        concatenated += seq
    return concatenated

def process_subfolder(subfolder: Path):
    """Concatenate CDSs from all .ffn files in subfolder and write one .fasta."""
    ffn_files = sorted(subfolder.glob("*.ffn"))
    if not ffn_files:
        print(f"⚠️ No .ffn files found in {subfolder}")
        return None

    subfolder_name = subfolder.name
    print(f"🧬 Processing subfolder: {subfolder_name}")

    all_cds = ""
    for ffn_file in ffn_files:
        all_cds += concatenate_cds_in_file(ffn_file)

    out_path = subfolder / f"{subfolder_name}_CDS.fasta"
    with open(out_path, "w") as f:
        f.write(f">{subfolder_name}_CDS\n")
        for i in range(0, len(all_cds), 70):
            f.write(all_cds[i:i+70] + "\n")

    print(f"  ✅ Saved concatenated CDS to: {out_path}")
    return out_path

def process_all_subfolders(root: Path):
    """Process all subfolders, then create a combined FASTA."""
    print(f"🔍 Searching for subfolders in: {root}\n")
    subfolders = [d for d in root.iterdir() if d.is_dir()]
    if not subfolders:
        print("⚠️ No subfolders found!")
        return

    concatenated_files = []
    for sub in subfolders:
        out = process_subfolder(sub)
        if out:
            concatenated_files.append(out)

    # Combine all subfolder fasta into one global file
    final_path = root / "all_concatenated_CDS.fasta"
    with open(final_path, "w") as fout:
        for fpath in concatenated_files:
            with open(fpath, "r") as fin:
                fout.write(fin.read().strip() + "\n")

    print(f"\n🌍 Final combined file created at:\n{final_path}")
    print(f"✅ Contains {len(concatenated_files)} concatenated sequences (one per subfolder).")

# === 🚀 Run the workflow ===

make_copy(SOURCE, DEST)
process_all_subfolders(DEST)